# VectorCam — Step 5c: Segment from ORIGINAL images (offline crop prep)

Plan: pull the 4 original zips, run U2Net segmentation on the FULL originals (label
included — segmentation ignores it), save tight mosquito crops + a new master to Drive.
This U2Net step is OFFLINE prep only; the phone never runs it.

Output: `data_seg.zip` + `master_seg.csv` in Drive, ready for training.

Set **GPU** runtime (segmentation is much faster on GPU).

## 0. Pull the 4 original zips + unzip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil
DRIVE='/content/drive/MyDrive/VectorCam'   # adjust if your zips are elsewhere
WORK='/content/orig'; os.makedirs(WORK, exist_ok=True)

zips=['0610.zip','0618.zip','0623.zip','kenya_01.zip']
for z in zips:
    src=os.path.join(DRIVE,z)
    if os.path.exists(src):
        with zipfile.ZipFile(src) as zf: zf.extractall(WORK)
        print('unzipped', z)
    else:
        print('MISSING', src)
print('\ntop-level under', WORK, ':', sorted(os.listdir(WORK)))

## 1. Rebuild master from originals
We already have master_cropped.csv (labels + metadata). We just need to point each
row to its ORIGINAL image on disk (not the 33%-cropped one). Match by ImageID.

In [ ]:
import pandas as pd, glob, re
# bring the existing master (labels/metadata) 
if not os.path.exists('/content/master_cropped.csv'):
    shutil.copy(os.path.join(DRIVE,'master_cropped.csv'),'/content/master_cropped.csv')
m = pd.read_csv('/content/master_cropped.csv')

# index every original image on disk by ImageID
IMG_RE = re.compile(r'image_(\d+)_[0-9a-f]+\.jpg$', re.I)
disk={}
for p in glob.glob(os.path.join(WORK,'**','*.jpg'), recursive=True):
    mt=IMG_RE.search(os.path.basename(p))
    if mt: disk[int(mt.group(1))]=p
print('original images found on disk:', len(disk))

m['orig_path']=m['ImageID'].map(disk)
missing=m['orig_path'].isna().sum()
print('rows without original found:', missing)
m=m[m['orig_path'].notna()].reset_index(drop=True)
print('usable rows:', len(m))

## 2. U2Net segmentation -> tight mosquito crop
Full U2Net (best quality) since this is offline. Fallback to center crop if it finds
too little. Preview on hard images first (incl. Kenya + the one that failed before).

In [ ]:
!pip -q install rembg onnxruntime-gpu
import numpy as np
from PIL import Image
from rembg import remove, new_session
import matplotlib.pyplot as plt
session=new_session('u2net')

def crop_center(rgb, frac=0.62):
    W,H=rgb.size; w,h=int(W*frac),int(H*frac); cx,cy=W//2,H//2
    return rgb.crop((cx-w//2,cy-h//2,cx+w//2,cy+h//2))

def crop_seg(pil_img, margin=0.18):
    rgb=pil_img.convert('RGB')
    out=remove(rgb, session=session)
    alpha=np.array(out)[:,:,3]; mask=alpha>30
    if mask.sum()<0.0004*mask.size: return crop_center(rgb)
    ys,xs=np.where(mask); H,W=mask.shape
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max()
    my,mx=int((y1-y0)*margin),int((x1-x0)*margin)
    y0=max(0,y0-my);y1=min(H,y1+my);x0=max(0,x0-mx);x1=min(W,x1+mx)
    # guard against degenerate boxes
    if (x1-x0)<0.03*W or (y1-y0)<0.03*H: return crop_center(rgb)
    return rgb.crop((x0,y0,x1,y1))

samp=pd.concat([m.sample(4,random_state=3), m[m.source=='field'].sample(4,random_state=5)]).drop_duplicates('ImageID')
fig,axes=plt.subplots(len(samp),2,figsize=(7,len(samp)*3))
for i,(_,r) in enumerate(samp.iterrows()):
    o=Image.open(r['orig_path']).convert('RGB')
    axes[i,0].imshow(o); axes[i,0].axis('off'); axes[i,0].set_title(f"{r['species']}|{r['drop']}",fontsize=7)
    axes[i,1].imshow(crop_seg(o)); axes[i,1].axis('off'); axes[i,1].set_title("segmented",fontsize=7)
plt.tight_layout(); plt.show()

## 3. Batch segment all -> /content/data_seg (run once preview looks good)

In [ ]:
from tqdm import tqdm
OUT='/content/data_seg'; os.makedirs(OUT, exist_ok=True)
paths=[]
for _,r in tqdm(m.iterrows(), total=len(m)):
    dst=os.path.join(OUT, f"image_{int(r['ImageID'])}.jpg")
    try: crop_seg(Image.open(r['orig_path']).convert('RGB')).save(dst,quality=95); paths.append(dst)
    except Exception: paths.append(None)
m['seg_path']=paths
print('segmented:', m.seg_path.notna().sum(), '| errors:', m.seg_path.isna().sum())
m.to_csv('/content/master_seg.csv', index=False)
shutil.make_archive('/content/data_seg','zip',OUT)
shutil.copy('/content/data_seg.zip', DRIVE)
shutil.copy('/content/master_seg.csv', DRIVE)
print('saved data_seg.zip + master_seg.csv to Drive')

## 4. Sanity grid

In [ ]:
chk=m[m.seg_path.notna()].sample(9, random_state=7)
fig,axes=plt.subplots(3,3,figsize=(10,11))
for ax,(_,r) in zip(axes.flatten(), chk.iterrows()):
    ax.imshow(Image.open(r['seg_path'])); ax.set_title(f"{r['species']}|{r['drop']}",fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## Next
Retrain on the segmented crops:
`python train.py --model efficientvit_b0 --master master_seg.csv --images data_seg`
(train.py needs the generic crop-column tweak — ask for it.)
Then compare Kenya macro-F1 vs the 0.67 loose baseline, and Grad-CAM to confirm the
heat is now on the mosquito.